# Hobby Corner RMS → Lightspeed Product Migration
**Analysis / dry-run only. This notebook does not write to Lightspeed.**

RMS on-hand quantity is used only as a migration-selection signal. Opening inventory is always 0 here; the physical inventory at cutover establishes actual Lightspeed stock.

In [ ]:
from google.colab import auth
auth.authenticate_user()

!pip -q install gspread pandas
!git clone -q -b product-migration-colab https://github.com/Tawgod/Hobby-Corner-Discord-Bot.git /content/hobby-corner || (cd /content/hobby-corner && git fetch -q && git checkout product-migration-colab && git pull -q)

import sys
sys.path.insert(0, '/content/hobby-corner/migration/product_migration')


In [ ]:
import google.auth
import gspread
import pandas as pd
from datetime import datetime, timezone
from analysis import (
    find_category_mapping_sheet,
    find_supplier_mapping_sheet,
    build_category_map,
    build_supplier_map,
    analyze_candidates,
    split_outputs,
    summary,
)

creds, _ = google.auth.default()
gc = gspread.authorize(creds)


## Source spreadsheets
The RMS candidate sheet is the export from RMS. The LS importer is authoritative for supplier/category destination mappings.

In [ ]:
RMS_CANDIDATE_SHEET = 'https://docs.google.com/spreadsheets/d/1_M4b0iD8EG3pn-cuR_uGZ3KoXL0bA1A_OpusCJboLSQ/edit'
LS_IMPORTER_SHEET = 'https://docs.google.com/spreadsheets/d/1-F1LpskwwkfiTJzQjpVmZHjOo7iOnCzcFw4CmOiN5kI/edit'

def workbook_frames(url):
    book = gc.open_by_url(url)
    frames = {}
    for ws in book.worksheets():
        values = ws.get_all_values()
        if not values:
            continue
        header_idx = next((i for i, r in enumerate(values) if any(str(x).strip() for x in r)), 0)
        headers = values[header_idx]
        data = values[header_idx + 1:]
        frames[ws.title] = pd.DataFrame(data, columns=headers)
    return frames

rms_tabs = workbook_frames(RMS_CANDIDATE_SHEET)
ls_tabs = workbook_frames(LS_IMPORTER_SHEET)
print('RMS tabs:', list(rms_tabs))
print('LS importer tabs:', list(ls_tabs))


In [ ]:
required = {'RMS_ItemID', 'RMS_SKU', 'RMS_Description', 'RMS_Department', 'RMS_Category'}
candidate_name, candidates = next((n, d) for n, d in rms_tabs.items() if required.issubset(set(d.columns)))

category_tab, category_df = find_category_mapping_sheet(ls_tabs)
supplier_tab, supplier_df, supplier_source_col, supplier_target_col = find_supplier_mapping_sheet(ls_tabs)

print('Candidate tab:', candidate_name)
print('Category mapping tab:', category_tab)
print('Supplier mapping tab:', supplier_tab, f'({supplier_source_col} -> {supplier_target_col})')


In [ ]:
category_map = build_category_map(category_df)
supplier_map = build_supplier_map(supplier_df, supplier_source_col, supplier_target_col)
result = analyze_candidates(candidates, category_map, supplier_map)
groups = split_outputs(result)

display(summary(result))
print('Unmapped categories:', int((~result['Category_Mapped']).sum()))
print('Unmapped suppliers:', int((~result['Supplier_Mapped']).sum()))
print('Opening inventory imported:', result['Opening_Inventory_To_Import'].sum())


## Write analysis results back to LS: importer
This updates only reporting tabs in the Google Sheet. It still does not write anything to Lightspeed.

In [ ]:
ls_book = gc.open_by_url(LS_IMPORTER_SHEET)

def get_or_create_tab(book, title, rows=1000, cols=30):
    try:
        return book.worksheet(title)
    except gspread.WorksheetNotFound:
        return book.add_worksheet(title=title, rows=rows, cols=cols)

run_ws = get_or_create_tab(ls_book, 'Migration Run Results', rows=500, cols=12)
review_ws = get_or_create_tab(ls_book, 'Migration Review', rows=max(len(groups['REVIEW']) + 10, 1000), cols=30)

run_time = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')
summary_row = [
    run_time,
    len(result),
    len(groups['READY']),
    len(groups['REVIEW']),
    len(groups['EXCLUDE']),
    int((~result['Category_Mapped']).sum()),
    int((~result['Supplier_Mapped']).sum()),
    float(result['Opening_Inventory_To_Import'].sum()),
    candidate_name,
    category_tab,
    supplier_tab,
]
headers = ['Run Time', 'Total', 'READY', 'REVIEW', 'EXCLUDE', 'Unmapped Categories', 'Unmapped Suppliers', 'Opening Inventory Imported', 'Candidate Tab', 'Category Mapping Tab', 'Supplier Mapping Tab']

if not run_ws.get_all_values():
    run_ws.append_row(headers, value_input_option='RAW')
run_ws.append_row(summary_row, value_input_option='RAW')

review_cols = [c for c in [
    'RMS_ItemID','RMS_SKU','RMS_Description','RMS_Department','RMS_Category','RMS_Supplier',
    'RMS_Quantity','LastSold','LastReceived','LS_Category_Path','LS_Supplier',
    'proposed_reorder_point','proposed_restock_level','review_reason'
] if c in result.columns]

review_out = groups['REVIEW'][review_cols].copy().fillna('')
review_ws.clear()
review_ws.update([review_out.columns.tolist()] + review_out.astype(str).values.tolist(), 'A1')

print('Saved run summary to: Migration Run Results')
print('Saved review queue to: Migration Review')


## Test-batch selection
This only selects candidates. It does **not** create products in Lightspeed.

In [ ]:
test_batch = result[
    (result['migration_status'] == 'READY') &
    result['Category_Mapped'] &
    result['Supplier_Mapped']
].head(10).copy()

display(test_batch[[c for c in [
    'RMS_ItemID','RMS_SKU','RMS_Description','LS_Category_Path','LS_Supplier',
    'proposed_reorder_point','proposed_restock_level'
] if c in test_batch.columns]])
